# 05 - Certification Workflow

Manages the table certification lifecycle:
- `uncertified` → `pending_review` (auto-triggered when all required tags present)
- `pending_review` → `certified` (manual approval or auto after review period)
- `certified` → `deprecated` (triggered by staleness + no owner response)

Also enforces review SLAs and escalates overdue certifications.

In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import (
    require_widget, uc_list_tables, uc_list_schemas,
    tables_to_spark, build_exempt_schemas,
    load_exemptions, is_exempt,
)
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("control_schema", "uc_hygiene")
dbutils.widgets.text("escalation_days", "7")
dbutils.widgets.text("review_window_days", "14")


catalog            = require_widget(dbutils, "catalog")
control_schema     = require_widget(dbutils, "control_schema")
escalation_days    = int(dbutils.widgets.get("escalation_days")    or "7")
review_window_days = int(dbutils.widgets.get("review_window_days") or "14")
control_fqn        = f"{catalog}.{control_schema}"

print(f"Control schema:      {control_fqn}")
print(f"Escalation days:     {escalation_days}")
print(f"Review window days:  {review_window_days}")
import time as _t; _task_start = _t.time()

In [0]:
# Function definitions moved to src/lib/common.py — imported in widget cell above.
from databricks.sdk import WorkspaceClient

_sdk = WorkspaceClient()


print("✅ lib.common loaded; SDK client ready.")


In [0]:
from datetime import date, timedelta

today = date.today()

# Step 1: Initialize new tables as uncertified via UC SDK
_table_rows = uc_list_tables(_sdk, [catalog], control_schema)  # certification runs against control catalog
_exemptions = load_exemptions(spark, catalog, control_schema)
_table_rows  = [r for r in _table_rows if not is_exempt(r["catalog_name"], r["schema_name"], r["table_name"], _exemptions)]
new_tables  = tables_to_spark(spark, _table_rows)
new_tables.createOrReplaceTempView("all_catalog_tables")

new_tables_to_init = spark.sql(f"""
SELECT t.table_catalog, t.table_schema, t.table_name
FROM all_catalog_tables t
LEFT JOIN {catalog}.{control_schema}.certification_state c
  ON t.table_catalog = c.catalog_name
  AND t.table_schema = c.schema_name
  AND t.table_name = c.table_name
WHERE c.table_name IS NULL
""")

new_count = new_tables_to_init.count()
print(f"New tables to initialize: {new_count}")

if new_count > 0:
    new_tables_to_init.createOrReplaceTempView("new_tables")
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.certification_state
    SELECT
      table_catalog, table_schema, table_name,
      'table'  AS asset_type,
      'uncertified' AS current_status,
      CURRENT_TIMESTAMP() AS status_changed_at,
      'system' AS status_changed_by,
      DATE_ADD(CURRENT_DATE(), review_window_days) AS next_review_date,
      NULL AS reviewer_email,
      'Auto-initialized by UC Steward' AS certification_notes
    FROM new_tables
    """)
    print(f"  Initialized {new_count} tables as 'uncertified'")


In [0]:
# Step 2: Auto-promote tables that have all required tags from uncertified → pending_review
fully_tagged = spark.sql(f"""
WITH required_tag_count AS (
  SELECT 
    catalog_name, schema_name, table_name,
    COUNT(DISTINCT tag_name) AS tag_count
  FROM {catalog}.information_schema.table_tags
  WHERE tag_name IN ('owner', 'domain', 'quality_tier')
  GROUP BY 1, 2, 3
  HAVING COUNT(DISTINCT tag_name) >= 3
)
SELECT cs.catalog_name, cs.schema_name, cs.table_name
FROM {catalog}.{control_schema}.certification_state cs
JOIN required_tag_count rtc
  ON cs.catalog_name = rtc.catalog_name
  AND cs.schema_name = rtc.schema_name
  AND cs.table_name = rtc.table_name
WHERE cs.current_status = 'uncertified'
""")

promote_count = fully_tagged.count()
if promote_count > 0:
    fully_tagged.createOrReplaceTempView("promote_candidates")
    spark.sql(f"""
    MERGE INTO {catalog}.{control_schema}.certification_state cs
    USING promote_candidates pc
    ON cs.catalog_name = pc.catalog_name 
       AND cs.schema_name = pc.schema_name 
       AND cs.table_name = pc.table_name
    WHEN MATCHED THEN UPDATE SET
      cs.current_status = 'pending_review',
      cs.status_changed_at = CURRENT_TIMESTAMP(),
      cs.status_changed_by = 'system',
      cs.next_review_date = DATE_ADD(CURRENT_DATE(), review_window_days),
      cs.certification_notes = 'All required tags present - ready for review'
    """)
    print(f"  Promoted {promote_count} tables to 'pending_review'")
else:
    print("  No tables ready for promotion")

In [0]:
# Step 3: Flag overdue reviews (pending_review past next_review_date)
overdue = spark.sql(f"""
SELECT catalog_name, schema_name, table_name, 
  next_review_date,
  DATEDIFF(CURRENT_DATE(), next_review_date) AS days_overdue
FROM {catalog}.{control_schema}.certification_state
WHERE current_status = 'pending_review'
  AND next_review_date < CURRENT_DATE()
""")

overdue_count = overdue.count()
print(f"\n⚠ Overdue reviews: {overdue_count}")
if overdue_count > 0:
    overdue.show(20, truncate=False)

In [0]:
# Step 4: Auto-deprecate tables that are both stale AND have unresolved findings for >escalation_days
auto_deprecate = spark.sql(f"""
SELECT DISTINCT cs.catalog_name, cs.schema_name, cs.table_name
FROM {catalog}.{control_schema}.certification_state cs
JOIN {catalog}.{control_schema}.scan_results sr
  ON cs.catalog_name = sr.catalog_name
  AND cs.schema_name = sr.schema_name
  AND cs.table_name = sr.table_name
WHERE sr.scan_type = 'staleness'
  AND sr.finding_severity = 'critical'
  AND sr.resolved_at IS NULL
  AND sr.scan_date < DATE_SUB(CURRENT_DATE(), {escalation_days})
  AND cs.current_status NOT IN ('deprecated')
""")

deprecate_count = auto_deprecate.count()
if deprecate_count > 0:
    auto_deprecate.createOrReplaceTempView("deprecate_candidates")
    spark.sql(f"""
    MERGE INTO {catalog}.{control_schema}.certification_state cs
    USING deprecate_candidates dc
    ON cs.catalog_name = dc.catalog_name 
       AND cs.schema_name = dc.schema_name 
       AND cs.table_name = dc.table_name
    WHEN MATCHED THEN UPDATE SET
      cs.current_status = 'deprecated',
      cs.status_changed_at = CURRENT_TIMESTAMP(),
      cs.status_changed_by = 'system',
      cs.certification_notes = 'Auto-deprecated: stale + unresolved critical findings'
    """)
    
    # Also set the tag on the actual table
    for row in auto_deprecate.collect():
        fqn = f"{row.catalog_name}.{row.schema_name}.{row.table_name}"
        try:
            spark.sql(f"SET TAG ON TABLE {fqn} certification_status = deprecated")
        except:
            pass
    
    print(f"  🚨 Auto-deprecated {deprecate_count} tables")
else:
    print("  No tables auto-deprecated")

print(f"""
========================================
  CERTIFICATION WORKFLOW COMPLETE  
========================================
  New tables initialized:   {new_count}
  Promoted to review:       {promote_count}
  Overdue reviews:          {overdue_count}
  Auto-deprecated:          {deprecate_count}
========================================
""")

In [0]:
# ── Summary & observability ──────────────────────────────────────────────────
total_tables_scanned = len(_table_rows)
print(f"""
{'='*52}
  CERTIFICATION WORKFLOW COMPLETE
{'='*52}
  Tables initialized (new):  {new_count}
  Promoted to pending_review: {promote_count}
  Overdue reviews:            {overdue_count}
  Auto-deprecated:            {deprecate_count}
  Escalation window:          {escalation_days}d
  Control schema: {catalog}.{control_schema}
{'='*52}
""")

total_findings = overdue_count + deprecate_count
try:
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.job_run_history VALUES (
      CURRENT_DATE(),
      'uc_hygiene_daily_governance',
      'p3_certification',
      'p3_remediation',
      'success',
      {total_tables_scanned},
      {total_findings},
      {total_findings},
      int(_t.time() - _task_start),
      'new={new_count} promoted={promote_count} overdue={overdue_count} deprecated={deprecate_count}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")